#  ⭐ `dim_dose`


In [1]:
%run ../../config/bootstrap.py


from pathlib import Path
from utils import get_project_root, normalize_dose

import re
import pandas as pd
from unidecode import unidecode

In [2]:
project_root = get_project_root() 

# 🥉 Bronze 

Raw Data
as is production, low quality

In [3]:
path = Path(project_root) / "data/01_bronze/anvisa/Medicamentos/Medicamentos.parquet"
bronze = pd.read_parquet(Path(project_root) / "data/01_bronze/anvisa/Medicamentos/Medicamentos.parquet")  
bronze['DOSE']= bronze['DOSE'].fillna('DESCONHECIDO')
bronze = (
    bronze[['DOSE']]
    .value_counts()
    .rename('FREQUENCIA_REGISTROS')
    .reset_index()
) 
bronze.columns

Index(['DOSE', 'FREQUENCIA_REGISTROS'], dtype='object')

In [4]:
bronze.DOSE.nunique()

10242

In [5]:
bronze.head(40)

,DOSE,FREQUENCIA_REGISTROS
0,DESCONHECIDO,362716
1,100 milligram (mg),10888
2,50 milligram (mg),10206
3,10 milligram (mg),8807
4,300 milligram (mg),7805
5,20 milligram (mg),7264
6,1 dosage form ({DF}),7168
7,40 milligram (mg),6389
8,500 milligram (mg),6151
9,1 gram (g),5719


In [6]:
bronze.to_csv(Path(project_root) / "eda/Medicamentos/Medicamentos_DOSE.csv", sep=";", index=False)

# 🥈 Silver

normalized data, medium quality


In [7]:
silver = normalize_dose(bronze, col="DOSE")

silver.columns

Index(['DOSE', 'FREQUENCIA_REGISTROS', 'DOSE_TIPO_VALOR', 'DOSE_TIPO_CHAVE',
       'DOSE_VALOR'],
      dtype='object')

In [8]:
silver[['DOSE_TIPO_CHAVE', 'DOSE_TIPO_VALOR']].drop_duplicates().sort_values(by='DOSE_TIPO_VALOR')

,DOSE_TIPO_CHAVE,DOSE_TIPO_VALOR
9504,30,24 hour (24.h)
7596,34,Maclagan unit ([mclg'U])
2821,22,degree Kelvin (K)
0,0,desconhecido
6,5,dosage form ({DF})
83,9,drop (1/12 millilitre) ([drp])
650,23,enzyme unit (U)
9,3,gram (g)
1426,16,gram metre (g.m)
599,17,gram per decilitre (g/dL)


In [9]:
silver.DOSE_VALOR.value_counts(dropna=False).head(10)

DOSE_VALOR
1.0      108
5.0       87
2.0       77
500.0     60
4.0       59
10.0      58
3.0       54
100.0     51
20.0      50
50.0      49
Name: count, dtype: int64

In [10]:
silver.query("DOSE_TIPO_CHAVE == 'desconhecido'").DOSE.value_counts(dropna=False).head(40)

Series([], Name: count, dtype: int64)

In [11]:
hist = silver

In [12]:
hist.to_parquet(Path(project_root) / "data/02_silver/hist_dim_dose/hist_dim_dose.parquet", index=False)

In [13]:
hist.to_csv(Path(project_root) / "data/02_silver/hist_dim_dose/hist_dim_dose_MANUAL.csv", sep=";", index=False)

# 🥇 Gold


In [14]:
hist.columns

Index(['DOSE', 'FREQUENCIA_REGISTROS', 'DOSE_TIPO_VALOR', 'DOSE_TIPO_CHAVE',
       'DOSE_VALOR'],
      dtype='object')

In [15]:
dim = hist[['DOSE_TIPO_CHAVE', 'DOSE_TIPO_VALOR']].sort_values(by='DOSE_TIPO_VALOR').drop_duplicates()
dim

,DOSE_TIPO_CHAVE,DOSE_TIPO_VALOR
9609,30,24 hour (24.h)
7596,34,Maclagan unit ([mclg'U])
3987,22,degree Kelvin (K)
4987,0,desconhecido
4118,5,dosage form ({DF})
10124,9,drop (1/12 millilitre) ([drp])
2360,23,enzyme unit (U)
6770,3,gram (g)
3814,16,gram metre (g.m)
1095,17,gram per decilitre (g/dL)


In [16]:
dim.to_csv(Path(project_root) / "data/03_gold/dim_dose/dim_dose.csv",sep=";", index=False)